In [1]:
import torch
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 랜덤 시드 고정
torch.manual_seed(777)

# GPU 사용 가능일 경우 랜덤 시드 고정
if device == 'cuda':
    torch.cuda.manual_seed_all(777)
    print('cuda')

cuda


In [ ]:
# 1. LR과 training epochs, batch size 선언
learning_rate = 0.001
training_epochs = 15
batch_size = 100

In [5]:
# 2. Dataloader 사용해서 데이터셋 선언
mnist_train = dsets.MNIST(root = 'MNIST_data/',
                          train = True,
                          transform = transforms.ToTensor(),
                          download = True)
mnist_test = dsets.MNIST(root = 'MNIST_data/',
                         train = False,
                         transform = transforms.ToTensor(),
                         download = True)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.42MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 160kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.54MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 17.9MB/s]


In [7]:
# Dataloader 사용해서 Batch size 지정해줘야함
data_loader = torch.utils.data.DataLoader(dataset = mnist_train,
                                          batch_size = batch_size,
                                          shuffle = True,
                                          drop_last = True)

In [8]:
# 3. 모델 설계
class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # first layer
        # 예상 shape (? ,28, 28, 1) 모르는건 input channel 
        # conv1로 마지막의 1을 32채널로
        # pooling 으로 28, 28을 14,14로
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1,32, kernel_size= 3, stride= 1, padding = 1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size= 2, stride= 2)
        )

        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size= 3, stride= 1, padding = 1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2, stride = 2)
        )

        self.fc = torch.nn.Linear(7*7*64, 10, bias = True) # 7*7*64 데이터를 10개의 output으로

        torch.nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        # 그리고 fc에 들어가기전 데이터를 펴줘야함
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [9]:
model = CNN().to(device)

In [10]:
criterion = torch.nn.CrossEntropyLoss().to(device) # Cost function 정의
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [11]:
total_batch = len(data_loader)
print('총 배치의 수 : {}'.format(total_batch))

총 배치의 수 : 600


In [12]:
# 위에서 batch_size = 100으로 선언하였는데
# total_batch size는 600이므로 100개짜리 뭉치가 600개 있다는 뜻
# 총 데이터는 60000개임

In [14]:
for epoch in range(training_epochs): # 15번 반복
    avg_cost = 0

    for X, Y in data_loader:
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()
        hypothesis = model(X) # 모델을 통하여 예측값 계산 Y는 실제값임
        cost = criterion(hypothesis, Y) # 예측값 hypothesis와 실제값 Y를 비교하여 cost 계산
        cost.backward()
        optimizer.step()

        avg_cost += cost / total_batch # 현재 배치 cost를 전체 배치수로 나누어서 누적

    print('[Epoch : {:>4}] cost = {:>.9}'.format(epoch + 1, avg_cost))

[Epoch :    1] cost = 0.0631814823
[Epoch :    2] cost = 0.0462594926
[Epoch :    3] cost = 0.0373750702
[Epoch :    4] cost = 0.0315814801
[Epoch :    5] cost = 0.0261291098
[Epoch :    6] cost = 0.0219022203
[Epoch :    7] cost = 0.0185954142
[Epoch :    8] cost = 0.0160958413
[Epoch :    9] cost = 0.0134205092
[Epoch :   10] cost = 0.0100731365
[Epoch :   11] cost = 0.0101282075
[Epoch :   12] cost = 0.00893703941
[Epoch :   13] cost = 0.00665497035
[Epoch :   14] cost = 0.00669868244
[Epoch :   15] cost = 0.00711953407


In [ ]:
# 테스트
with torch.no_grad():
    X_test = mnist_test.test_data.view(len(mnist_test), 1, 28, 28).float().to(device)
    Y_test = mnist_test.test_labels.to(device)

    prediction = model(X_test)

    correct_prediction = torch.argmax(prediction, 1) == Y_test 

    accuracy = correct_prediction.float().mean()
    print('Accuracy : ', accuracy.item())

Accuracy :  0.9839999675750732
